<a href="https://colab.research.google.com/github/Ranesshtallapelly/Data-analysis-python/blob/main/dapweek11.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install python-docx

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 5.4 MB/s eta 0:00:00


In [5]:
import math, numpy as np, pandas as pd, matplotlib.pyplot as plt, json
import os # Import the os module
np.random.seed(42)

# Create the directory if it doesn't exist
output_dir = '/mnt/data/'
os.makedirs(output_dir, exist_ok=True)

# Parameters
n = 120
p = 0.92
lam = 14

def binomial_pmf(n,k,p):
    return math.comb(n,k) * (p**k) * ((1-p)**(n-k))

def poisson_pmf(lam,k):
    return (math.exp(-lam) * (lam**k)) / math.factorial(k)

# Binomial PMF/CDF
bin_rows=[]
cum=0.0
for k in range(0,n+1):
    pmf = binomial_pmf(n,k,p)
    cum += pmf
    bin_rows.append({"k":k,"pmf":pmf,"cdf":cum})
bin_df = pd.DataFrame(bin_rows)

prob_eq_110 = binomial_pmf(n,110,p)
prob_ge_115 = sum(binomial_pmf(n,k,p) for k in range(115,n+1))
q85 = int(bin_df[bin_df["cdf"]>=0.85]["k"].iloc[0])

sim_bins = np.random.binomial(n,p,size=2000)
sim_bin_df = pd.DataFrame({"hour":np.arange(1,2001),"correct_flags":sim_bins})

# Binomial plots (focus around mean ± 6sd)
mean_bin = n*p
sd_bin = (n*p*(1-p))**0.5
k_min = max(0,int(mean_bin-6*sd_bin))
k_max = min(n,int(mean_bin+6*sd_bin))
focus_df = bin_df[(bin_df.k>=k_min)&(bin_df.k<=k_max)]
plt.figure(figsize=(10,5))
plt.bar(focus_df["k"], focus_df["pmf"])
plt.axvline(mean_bin, color='red', linestyle='--', label=f"Mean={mean_bin:.2f}")
plt.legend()
plt.savefig(f"{output_dir}week11_binomial_pmf.png"); plt.close()

plt.figure(figsize=(10,5))
plt.step(bin_df["k"], bin_df["cdf"], where='mid')
plt.axvline(q85, color='green', linestyle='--', label=f"85th pct={q85}")
plt.legend()
plt.savefig(f"{output_dir}week11_binomial_cdf.png"); plt.close()

# Poisson PMF/CDF
po_rows=[]
cum=0.0
for k in range(0,31):
    pmf = poisson_pmf(lam,k)
    cum += pmf
    po_rows.append({"k":k,"pmf":pmf,"cdf":cum})
po_df = pd.DataFrame(po_rows)

prob_eq_20 = poisson_pmf(lam,20)
prob_le_12 = po_df[po_df.k<=12]["pmf"].sum()
q97 = int(po_df[po_df["cdf"]>=0.97]["k"].iloc[0])

sim_po = np.random.poisson(lam,size=48)
sim_po_df = pd.DataFrame({"hour":np.arange(1,49),"landings":sim_po})

plt.figure(figsize=(10,5))
plt.bar(po_df["k"], po_df["pmf"])
plt.savefig(f"{output_dir}week11_poisson_pmf.png"); plt.close()

plt.figure(figsize=(10,5))
plt.step(po_df["k"], po_df["cdf"], where='mid')
plt.savefig(f"{output_dir}week11_poisson_cdf.png"); plt.close()

# Save CSVs and combined
bin_df.to_csv(f"{output_dir}week11_binomial_pmf_cdf.csv", index=False)
sim_bin_df.to_csv(f"{output_dir}week11_binomial_sim_2000.csv", index=False)
po_df.to_csv(f"{output_dir}week11_poisson_pmf_cdf.csv", index=False)
sim_po_df.to_csv(f"{output_dir}week11_poisson_sim_48.csv", index=False)

# Combined
combined_rows=[]
for _,r in bin_df.iterrows():
    combined_rows.append({"k":int(r.k),"pmf":r.pmf,"cdf":r.cdf,"value":None,"source":"binomial_pmf"})
for _,r in po_df.iterrows():
    combined_rows.append({"k":int(r.k),"pmf":r.pmf,"cdf":r.cdf,"value":None,"source":"poisson_pmf"})
for _,r in sim_bin_df.iterrows():
    combined_rows.append({"k":int(r.hour),"pmf":None,"cdf":None,"value":int(r.correct_flags),"source":"binomial_sim"})
for _,r in sim_po_df.iterrows():
    combined_rows.append({"k":int(r.hour),"pmf":None,"cdf":None,"value":int(r.landings),"source":"poisson_sim"})
combined_df = pd.DataFrame(combined_rows)
combined_df.to_csv(f"{output_dir}week11_all_combined.csv", index=False)

# Summary
summary = {
  "P(X=110)": prob_eq_110,
  "P(X>=115)": prob_ge_115,
  "qbinom(0.85)": q85,
  "P(X=20) (poisson)": prob_eq_20,
  "P(X<=12) (poisson)": prob_le_12,
  "qpois(0.97)": q97
}
with open(f"{output_dir}week11_summary.json","w") as f:
    json.dump(summary,f,indent=2)